In [ ]:
!pip3 install jiwer
!pip3 install evaluate

In [ ]:
import torch
import torchaudio
import gc
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2CTCTokenizer, HubertForCTC, Wav2Vec2ForCTC, Wav2Vec2Processor
from huggingface_hub import login

login(token="REMOVED_SECRET")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

main_seed=42

model_name = "facebook/hubert-base-ls960"
model_language = "mansi"

path = "/kaggle/input/mansi-language/"

dataset = load_dataset('csv', data_files=path+'dataset.csv')

In [ ]:
import json

transcripts = dataset['train']['sentence']

translation_table = dict.fromkeys(map(ord, '":;«¬»–—…\'̄().?!-,0123456789'), None)
transcripts = [line.translate(translation_table).lower().strip() for line in transcripts]

all_text = " ".join(transcripts)
vocab = list(set(all_text))

vocab_dict = {char: i for i, char in enumerate(vocab)}
vocab_dict['|'] = vocab_dict[' ']
del vocab_dict[' ']
vocab_dict["[PAD]"] = len(vocab_dict)
vocab_dict["[UNK]"] = len(vocab_dict)

with open('vocab.json', 'w') as vocab_file:
    json.dump(vocab_dict, vocab_file)

tokenizer = Wav2Vec2CTCTokenizer("./vocab.json", unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|")

In [ ]:
tokenizer.get_vocab()

In [ ]:
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)

processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

def feature_extract(batch):
    speech_array, sampling_rate = torchaudio.load(path+batch['file'])
    speech = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=16000)(speech_array).squeeze().numpy()
    batch["input_values"] = feature_extractor(speech, sampling_rate=16000).input_values[0]
    return batch
    

def tokenize(batch):
    text = batch['sentence'].translate(translation_table).lower().strip()
    with processor.as_target_processor():
        batch["labels"] = processor(text).input_ids

    return batch

dataset = dataset.map(feature_extract, num_proc=4)

dataset = dataset.map(tokenize, num_proc=4)

dataset.set_format(type='torch', columns=['input_values', 'labels'])
dataset = dataset['train'].train_test_split(test_size=0.05, seed=main_seed)

dataset["train"].features

In [ ]:
# import IPython.display as ipd
# import numpy as np
# import random

# rand_int = random.randint(0, len(dataset["train"]))

# print(tokenizer.decode(dataset["train"]["labels"][rand_int]))
# ipd.Audio(data=np.asarray(dataset["train"]["input_values"][rand_int]), autoplay=True, rate=16000)


In [ ]:
import torch

from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union

@dataclass
class DataCollatorCTCWithPadding:
    """
    Data collator that will dynamically pad the inputs received.
    Args:
        processor (:class:`~transformers.Wav2Vec2Processor`)
            The processor used for proccessing the data.
        padding (:obj:`bool`, :obj:`str` or :class:`~transformers.tokenization_utils_base.PaddingStrategy`, `optional`, defaults to :obj:`True`):
            Select a strategy to pad the returned sequences (according to the model's padding side and padding index)
            among:
            * :obj:`True` or :obj:`'longest'`: Pad to the longest sequence in the batch (or no padding if only a single
              sequence if provided).
            * :obj:`'max_length'`: Pad to a maximum length specified with the argument :obj:`max_length` or to the
              maximum acceptable input length for the model if that argument is not provided.
            * :obj:`False` or :obj:`'do_not_pad'` (default): No padding (i.e., can output a batch with sequences of
              different lengths).
        max_length (:obj:`int`, `optional`):
            Maximum length of the ``input_values`` of the returned list and optionally padding length (see above).
        max_length_labels (:obj:`int`, `optional`):
            Maximum length of the ``labels`` returned list and optionally padding length (see above).
        pad_to_multiple_of (:obj:`int`, `optional`):
            If set will pad the sequence to a multiple of the provided value.
            This is especially useful to enable the use of Tensor Cores on NVIDIA hardware with compute capability >=
            7.5 (Volta).
    """

    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True
    max_length: Optional[int] = None
    max_length_labels: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None
    pad_to_multiple_of_labels: Optional[int] = None

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need
        # different padding methods
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )
        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(
                label_features,
                padding=self.padding,
                max_length=self.max_length_labels,
                pad_to_multiple_of=self.pad_to_multiple_of_labels,
                return_tensors="pt",
            )

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        batch["labels"] = labels

        return batch


data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

In [ ]:
import evaluate
import numpy as np

metric_wer = evaluate.load("wer")
metric_cer = evaluate.load("cer")

def compute_metrics(pred):
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True, group_tokens=False)
    
    random_idx = np.random.randint(0, len(pred_ids))
    
    print("Label IDs:", label_ids[random_idx])
    print("Predicted IDs:", pred_ids[random_idx])
    print("Label text:", label_str[random_idx])
    print("Predicted text:", pred_str[random_idx])

    wer = 100 * metric_wer.compute(predictions=pred_str, references=label_str)
    cer = 100 * metric_cer.compute(predictions=pred_str, references=label_str)

    print("norm_wer: " + str(wer))
    print("norm_cer: " + str(cer))
    
    return {"norm_wer": wer, "norm_cer": cer}

In [ ]:
model = HubertForCTC.from_pretrained(
    model_name ,
    ctc_loss_reduction="mean", 
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size = processor.tokenizer.vocab_size
)

In [ ]:
from transformers import TrainingArguments, Trainer

model.freeze_feature_extractor()

training_args = TrainingArguments(
    output_dir="./hubert-mansi",
    overwrite_output_dir=True,
    save_total_limit=2,
    
    # learning_rate=1e-4,
    # weight_decay=0.005,
    # optim='adafactor',
    group_by_length=True,
    gradient_checkpointing=True,
    gradient_accumulation_steps=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    max_steps=15000,
    save_strategy="steps",
    save_steps=500,
    eval_strategy="steps",
    eval_steps=500,

    seed=main_seed,
    data_seed=main_seed,
    fp16=torch.cuda.is_available(),
    fp16_full_eval=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="norm_wer",
    greater_is_better=False,
    report_to=["tensorboard"],
    push_to_hub=True,
    hub_private_repo=True,
)

trainer = Trainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

trainer.train()